# Galaxy Classifier - Kaggle Training Notebook

Trains a ResNet18-based classifier on the Galaxy Zoo dataset.

**Before running**: attach the "Galaxy Zoo - The Galaxy Challenge" dataset/competition to this notebook (Add Data panel), and set Accelerator to GPU in notebook Settings.

Adjust `IMAGES_DIR` and `LABELS_CSV` below to match the actual paths under `/kaggle/input/` once the dataset is attached (check the file browser on the right).

In [ ]:
!pip install -q torchvision tqdm scikit-learn matplotlib pandas pillow

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import models, transforms
from torchvision.models import ResNet18_Weights
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

In [ ]:
# --- ADJUST THESE PATHS to match /kaggle/input/... once dataset is attached ---
IMAGES_DIR = '/kaggle/input/galaxy-zoo-the-galaxy-challenge/images_training_rev1/images_training_rev1'
LABELS_CSV = '/kaggle/input/galaxy-zoo-the-galaxy-challenge/training_solutions_rev1/training_solutions_rev1.csv'
OUTPUT_DIR = '/kaggle/working/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLASS_NAMES = ['elliptical', 'spiral', 'other']
LABEL_COLUMNS = ['Class1.1', 'Class1.2', 'Class1.3']

# Start small to sanity-check the pipeline runs end to end, then increase.
SUBSET_FRAC = 0.2
EPOCHS = 5
BATCH_SIZE = 32
LR = 1e-3

In [ ]:
class GalaxyZooDataset(Dataset):
    def __init__(self, images_dir, labels_csv, transform=None, subset_frac=None, seed=42):
        self.images_dir = images_dir
        self.transform = transform

        df = pd.read_csv(labels_csv)
        if subset_frac is not None:
            df = df.sample(frac=subset_frac, random_state=seed).reset_index(drop=True)

        df['label'] = np.argmax(df[LABEL_COLUMNS].values, axis=1)
        self.galaxy_ids = df['GalaxyID'].astype(str).tolist()
        self.labels = df['label'].tolist()

    def __len__(self):
        return len(self.galaxy_ids)

    def __getitem__(self, idx):
        galaxy_id = self.galaxy_ids[idx]
        label = self.labels[idx]
        img_path = os.path.join(self.images_dir, f'{galaxy_id}.jpg')
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

    def get_class_weights(self):
        """Inverse-frequency class weights to counteract imbalance
        (the 'other' class is naturally very rare in Galaxy Zoo)."""
        counts = np.bincount(self.labels, minlength=len(CLASS_NAMES))
        counts = np.maximum(counts, 1)
        weights = counts.sum() / (len(CLASS_NAMES) * counts)
        return weights.astype(np.float32)

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

full_dataset = GalaxyZooDataset(IMAGES_DIR, LABELS_CSV, transform=eval_transform, subset_frac=SUBSET_FRAC)

# Stratified split so the rare 'other' class is represented in both
# train and val (a plain random split previously left only 2 'other'
# examples in validation, making that class's metrics meaningless).
indices = np.arange(len(full_dataset))
train_idx, val_idx = train_test_split(
    indices, test_size=0.15, stratify=full_dataset.labels, random_state=42
)

train_dataset_aug = GalaxyZooDataset(IMAGES_DIR, LABELS_CSV, transform=train_transform, subset_frac=SUBSET_FRAC)
train_ds = Subset(train_dataset_aug, train_idx)
val_ds = Subset(full_dataset, val_idx)

train_labels = [full_dataset.labels[i] for i in train_idx]
val_labels = [full_dataset.labels[i] for i in val_idx]
print(f'Train size: {len(train_ds)} | Val size: {len(val_ds)}')
print(f'Train class counts: {np.bincount(train_labels, minlength=len(CLASS_NAMES))}')
print(f'Val class counts:   {np.bincount(val_labels, minlength=len(CLASS_NAMES))}')

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
def build_model(num_classes=3, unfreeze_last_block=True):
    weights = ResNet18_Weights.IMAGENET1K_V1
    model = models.resnet18(weights=weights)
    for param in model.parameters():
        param.requires_grad = False
    if unfreeze_last_block:
        for param in model.layer4.parameters():
            param.requires_grad = True
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model

model = build_model(num_classes=len(CLASS_NAMES)).to(device)

# Class-weighted loss so the model doesn't just ignore the rare 'other' class
class_weights = torch.tensor(full_dataset.get_class_weights()).to(device)
print('Class weights:', dict(zip(CLASS_NAMES, class_weights.tolist())))
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)

In [ ]:
best_val_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [train]'):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)

    train_loss = running_loss / len(train_ds)

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [val]'):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total if total > 0 else 0.0
    print(f'Epoch {epoch+1}: train_loss={train_loss:.4f} val_acc={val_acc:.4f}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, 'best_model.pt'))
        print('  -> New best val_acc, model saved.')

print(f'Training complete. Best val_acc: {best_val_acc:.4f}')

## Evaluate + Confusion Matrix

In [ ]:
model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, 'best_model.pt')))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(CLASS_NAMES)))
ax.set_yticks(range(len(CLASS_NAMES)))
ax.set_xticklabels(CLASS_NAMES, rotation=45)
ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix')
for i in range(len(CLASS_NAMES)):
    for j in range(len(CLASS_NAMES)):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center')
fig.colorbar(im)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'))
plt.show()

## Next steps

- Download `best_model.pt` and `confusion_matrix.png` from `/kaggle/working/outputs` (Output panel on the right) to bring back into your local repo.
- Once the pipeline is validated end-to-end at `SUBSET_FRAC=0.2`, increase toward `1.0` for a fuller training run.
- Look at a handful of misclassified images to include in the portfolio write-up.